# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://3027bb85-6b59-486c-9381-e5ca30e5bc04.eu-central-1-0.aws.cloud.qdrant.io:6333


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [3]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "C:/Users/doyoo/smu-ai-service-bootcamp/rag-system/datasets/붙임2_2026학년도_2학기_학과별시간표(2026.08.19.).pdf"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 98개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 342자
평균 페이지 길이: 1891자

첫 페이지 내용 미리보기:
               2026학년도2학기학과별시간표(공지용)


인문사회과학대학
                                  실습             강의시간    분 No 학년이수  학수번호      교과목명     학점이론     교양영역                  담당교수        비고
      구분                       시간 시간                  (강의실)    반

  1   3 1전선 HARF0001 취업과창업(인문사회)         1   1   0   ...


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 98
  - Child chunk 수: 652
  - 평균 chunk/page: 6.7

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: page_1
  Page: 1
  Length: 327자
  Content: 2026학년도2학기학과별시간표(공지용)


인문사회과학대학
                                  실습             강의시간    분 No 학년이수 ...

Chunk 2:
  Parent ID: page_2
  Page: 2
  Length: 244자
  Content: 2026학년도2학기학과별시간표(공지용)


인문사회과학대학인문콘텐츠학부역사콘텐츠전공
                                  실습             강의시간...

Chunk 3:
  Parent ID: page_2
  Page: 2
  Length: 340자
  Content: 1   1 1전선 HAAA6005 한국사사료강독과DB활용       3   1   2                  수7,8,9(R107)         1   정다함  1학년전용...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://3027bb85-6b59-486c-9381-e5ca30e5bc04.eu-central-1-0.aws.cloud.qdrant.io:6333


In [6]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "SMU_2026_2_time"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'SMU_2026_2_time' 생성 완료

652개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [7]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 98개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_1', 'page_2', 'page_3', 'page_4', 'page_5']


## 5. Parent Document Retriever 구현

In [8]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [9]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "난 소프트웨어학과 2학년이 들을 수 있는 수업은 뭐야?"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 난 소프트웨어학과 2학년이 들을 수 있는 수업은 뭐야?


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 40
  Parent ID: page_40
  길이: 396자
  내용: 11   2 1전선 HAEA0002 컴퓨터구조                 3   3   0                  목7,8,9(G208)         2   민승욱

  12   2 1전선 HAEA0003 시스템소프트웨어            3   1   2                  월5,6(G207) 수7(G207)  1   손성훈

  13   2 1전선 HAEA0003 시스템소프트웨어            3   1   2                 월7(G207) 수5,6(G207)  2   손성훈

  14   2 1전선 HAEA0010 알고리즘                   3   1   2                  화1,2,3(G207)         1   민경하

Chunk 2:
  페이지: 37
  Parent ID: page_37
  길이: 280자
  내용: 1학년전용수업(재수강대상자  1   1 교선  HALR1232 교양과인성(휴먼AI공학전공)    1   1   0 일반(인성과리더십) 목6(G309)            1   유지연                                                                  계당교양교육원별도신청)
  2   1 1전선  HAIE0004 C프로그래밍2               3   1   2                  수7,8,9(G102)         1   이의철


[2] Parent Document 검색 결과
-----------------------------------

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은 상명대에서 12년간 종사한 모든 학사에 대한 공지나 자료를 꿰뚫고 있는 교직원입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하고 가독성 있고 깔끔한 답변을 하세요.
또한, 답변에 참고한 문서의 출처와 페이지 번호를 명시하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content

print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [12]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "소프트웨어학과 2학년이 들을 수 있는 수업은 뭐야?",
    "월요일, 금요일 제외하고 들을 수 있는 수업 목록 알려줘",
    "균형이라고 적혀있는 수업 목록 알려줘"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 소프트웨어학과 2학년이 들을 수 있는 수업은 뭐야?



소프트웨어학과 **2학년** 기준으로 수강 가능한 과목은, 참고자료에 따르면 다음과 같습니다.  
(※ 다만, 제공해주신 자료는 **휴먼AI공학전공 시간표** 페이지라서, **소프트웨어학과 전체 개설과목**이 아니라 **해당 전공 기준**으로 확인한 내용입니다.)

## 2학년 수강 가능 과목
1. **컴퓨터네트워크**  
   - 학수번호: **HAIE0013**  
   - 분반: **1, 2**  
   - 강의시간: **화4,5,6(G305)** / **목4,5,6(G305)**  
   - 담당교수: **정진우**

2. **알고리즘**  
   - 학수번호: **HAIE0014**  
   - 분반: **1**  
   - 강의시간: **화1,2,3(G102)**  
   - 담당교수: **오영민**

3. **인공지능개론**  
   - 학수번호: **HAIE0015**  
   - 분반: **1**  
   - 강의시간: **목1,2,3(G102)**  
   - 담당교수: **윤희남**

4. **빅데이터분석**  
   - 학수번호: **HAIE0023**  
   - 분반: **1, 2**  
   - 강의시간: **목7,8,9(G311)** / **화4,5,6(G311)**  
   - 담당교수: **임좌상 / 유지연**

5. **웹프로그래밍**  
   - 학수번호: **HAIE0026**  
   - 분반: **1**  
   - 강의시간: **수7,8,9(G305)**  
   - 담당교수: **미정**

6. **데이터베이스**  
   - 학수번호: **HAIE0037**  
   - 분반: **1, 2**  
   - 강의시간: **수4,5,6(G102)** / **목7,8,9(G310)**  
   - 담당교수: **김동근**

---

## 참고
- **출처:** 붙임2_2026학년도_2학기_학과별시간표(2026.08.19.).pdf  
- **페이지:** 37

원하시면 제가 이어서  
**“2학년이 들을 수 있는 과목 중 필수/전선 구분”**까지 깔끔하게 정리해드릴게요.


질문: 월요일, 금요일 제외하고 들을 수 있는 수업 목록 알려줘



아래는 **월요일, 금요일 수업이 없는 과목**만 추려드린 목록입니다.  
즉, **화요일 / 수요일 / 목요일**에만 배정된 수업입니다.

---

## 1) 역사콘텐츠전공
**출처:** 붙임2_2026학년도_2학기_학과별시간표(2026.08.19.).pdf, **p.2**

- **HAAA9230 서양고대사**  
  - 화 7, 8, 9
- **HAAA9231 서양중세사**  
  - 수 7, 8, 9
- **HAAA9232 디지털문화유산**  
  - 화 3, 수 5, 6
- **HAAA9237 전공체험(역사콘텐츠전공)**  
  - 목 7, 8, 9

---

## 2) 의류학과
**출처:** 붙임2_2026학년도_2학기_학과별시간표(2026.08.19.).pdf, **p.67**

- **HALR1215 교양과인성(의류학)**  
  - 금 수업이므로 제외
- **HACK0020 패션리테일테크**  
  - 수 7, 8, 9 / 목 7, 8, 9
- **HACK0011 컨셉개발**  
  - 수 4, 5, 6
- **HACK0027 비주얼머천다이징**  
  - 목 5, 6 / 목 0
- **HACK9227 패션트렌드와소재**  
  - 목 3, 4 / 금 5 → 금 포함이므로 제외
- **HACK0015 드레이핑2**  
  - 목 4, 5, 6
- **HACK9009 글로벌패션비즈니스**  
  - 목 7, 8, 9
- **HACK9013 패션소재기획**  
  - 화 4, 5, 6
- **HALF7021 문화와복식**  
  - 금 10, 11, 12 → 금 포함이므로 제외

---

## 최종 정리: 월·금 제외하고 가능한 수업
### 역사콘텐츠전공
- 서양고대사
- 서양중세사
- 디지털문화유산
- 전공체험(역사콘텐츠전공)

### 의류학과
- 패션리테일테크
- 컨셉개발
- 비주얼머천다이징
- 드레이핑2
- 글로벌패션비즈니스
- 패션소재기획

---

원하시면 제가 이어서  
**“월/금 제외 + 시간대가 겹치지 않게”** 조합까지 짜드릴게요.


질문: 균형이라고 적혀있는 수업 목록 알려줘



아래는 **제공해주신 2026학년도 2학기 학과별시간표**에서 **교양영역이 “균형”으로 표시된 수업** 목록입니다.  
(페이지 78~79 기준)

---

## 균형 교과목 목록

### 페이지 78
1. **HALF9031 현대사회와심리학**  
   - 분반: 1  
   - 교양영역: 균형(사회)  
   - 강의시간: 월 2,3,4  
   - 담당교수: 한규은  

2. **HALF9041 과학의철학적이해**  
   - 분반: 1  
   - 교양영역: 균형(자연)  
   - 강의시간: 월 5,6 / 금 2  
   - 담당교수: 이청호  

3. **HALF9061 현대미술의이해**  
   - 분반: 1  
   - 교양영역: 균형(예술)  
   - 강의시간: 금 1,2,3  
   - 담당교수: 미정  

4. **HALF9245 현대사회와인간**  
   - 분반: 1  
   - 교양영역: 균형(사회)  
   - 강의시간: 목 7,8,9  
   - 담당교수: 이영면  

5. **HALF9266 현대정치의이해**  
   - 분반: 1  
   - 교양영역: 균형(사회)  
   - 강의시간: 화 4 / 수 5,6  
   - 담당교수: 박정호  

6. **HALF9279 긍정심리와행복탐구**  
   - 분반: 1  
   - 교양영역: 일반(인간과사회)  
   - ※ 균형 아님

7. **HALF9301 셀프리더십과자기계발**  
   - 분반: 1  
   - 교양영역: 일반(인성과리더십)  
   - ※ 균형 아님

8. **HALF9021 논리와비판적사고**  
   - 교양영역: 일반(역사와철학)  
   - ※ 균형 아님

9. **HALF9284 범죄와사회**  
   - 교양영역: 일반(법과정치)  
   - ※ 균형 아님

10. **HALF9287 AI융합형영상콘텐츠기획/제작**  
   - 교양영역: 일반(정보기술과산업)  
   - ※ 균형 아님

---

### 페이지 79
1. **HALF9319 창의적프로그래밍입문**  
   - 분반: 1  
   - 교양영역: 균형(공학)  
   - 강의시간: 화 7,8,9  
   - 담당교수: 민경하  

2. **HALF9320 리터러시탐구와응용**  
   - 분반: 1, 2  
   - 교양영역: 균형(사회)  
   - 강의시간: 화 5,6 / 화 7,8  
   - 담당교수: 안형환  

3. **HALF9321 미래생활과화학**  
   - 분반: 1  
   - 교양영역: 균형(자연)  
   - 강의시간: 수 7,8,9  
   - 담당교수: 강상욱  

4. **HALF9326 창의적사고의프레임워크**  
   - 분반: 1  
   - 교양영역: 균형(사회)  
   - 강의시간: 화 7,8,9  
   - 담당교수: 유상건  

5. **HALF9329 미래사회와디지털기술**  
   - 분반: 1  
   - 교양영역: 균형(공학)  
   - 강의시간: 화 7,8,9  
   - 담당교수: 유지연  

6. **HALF9338 명저읽기(문학)**  
   - 분반: 1  
   - 교양영역: 균형(인문)  
   - 강의시간: 수 7,8,9  
   - 담당교수: 강옥희  

7. **HALF9340 세계문화교류**  
   - 분반: 1  
   - 교양영역: 균형(인문)  
   - 강의시간: 수 7,8,9  
   - 담당교수: 정유선  

8. **HALF9343 세계종교와문화**  
   - 분반: 1  
   - 교양영역: 균형(사회)  
   - 강의시간: 목 7,8,9  
   - 담당교수: 김일림  

9. **HALF9356 현대미술사와이론**  
   - 분반: 1  
   - 교양영역: 균형(예술)  
   - 강의시간: 월 10,11,12  
   - 담당교수: 오경은  
   - 비고: E-러닝교과목 / 3-4학년만 수강 가능

10. **HALF9358 휴먼커뮤니케이션**  
    - 분반: 1  
    - 교양영역: 균형(인문)  
    - 강의시간: 화 10,11,12  
    - 담당교수: 박재현  
    - 비고: E-러닝교과목 / 3-4학년만 수강 가능

11. **HALF9362 정량적사고**  
    - 분반: 1  
    - 교양영역: 균형(자연)  
    - 강의시간: 목 7,8,9  
    - 담당교수: 배윤한  

---

## 정리
**“균형”으로 표시된 수업**은 위 과목들입니다.  
원하시면 제가 다음 단계로도 정리해드릴 수 있어요:

- **균형 과목만 표로 보기**
- **분야별(사회/자연/인문/공학/예술)로 묶어서 보기**
- **학수번호만 빠르게 추려서 보기**

---

### 참고 출처
- **붙임2_2026학년도_2학기_학과별시간표(2026.08.19.).pdf**
  - **페이지 78**
  - **페이지 79**

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합